# Stage 11D — best.pt checkpoint sensitivity

Evaluation only. No new training. Uses the six completed validation-selected `best.pt` checkpoints under the frozen Stage 6 evaluation settings.

In [ ]:
from pathlib import Path
from google.colab import drive
import shutil, subprocess
drive.mount('/content/drive', force_remount=False)
DRIVE=Path('/content/drive/MyDrive')
RUN_ROOT=DRIVE/'Stage5_Baselines'
STAGE6=DRIVE/'Stage6_Heldout_Evaluation'
OUT=DRIVE/'Stage11D_Best_Checkpoint_Sensitivity'
OUT.mkdir(parents=True,exist_ok=True)
if not shutil.which('nvidia-smi'): raise RuntimeError('Select T4 GPU.')
subprocess.run(['nvidia-smi'],check=True)
print('STAGE11D_GPU: PASS')

In [ ]:
import hashlib, zipfile, shutil
PARTS=[DRIVE/'Stage5_Frozen_h9_v2_Data_Bundle.zip.part_00',DRIVE/'Stage5_Frozen_h9_v2_Data_Bundle.zip.part_01',DRIVE/'Stage5_Frozen_h9_v2_Data_Bundle.zip.part_02']
BUNDLE=Path('/content/Stage5_Frozen_h9_v2_Data_Bundle.zip')
EXPECTED='dd8ce20f2650848133ce4323b00139684b0d4ac8b7db618e8df469244221cee9'
h=hashlib.sha256()
with BUNDLE.open('wb') as out:
    for p in PARTS:
        with p.open('rb') as f:
            for ch in iter(lambda:f.read(8*1024*1024),b''):
                out.write(ch); h.update(ch)
assert h.hexdigest()==EXPECTED
DATA=Path('/content/stage11d_data')
if DATA.exists(): shutil.rmtree(DATA)
DATA.mkdir()
with zipfile.ZipFile(BUNDLE) as z: z.extractall(DATA)
Y_DATA=DATA/'h9_v2/Y-PPE-h9-v2/data.yaml'
C_DATA=DATA/'h9_v2/Construction-PPE-h9-v2/data.yaml'
print('STAGE11D_DATA: PASS')

In [ ]:
import subprocess, shutil, sys
REPO=Path('/content/Y-PPE-Detection-RFDETR')
if REPO.exists(): shutil.rmtree(REPO)
subprocess.run(['git','clone','--branch','stage11-major-revision','--single-branch','https://github.com/EZZALDEEN049/Y-PPE-Detection-RFDETR.git',str(REPO)],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','ultralytics==8.4.156','PyYAML==6.0.3','pandas'],check=True)
print('STAGE11D_CODE: PASS')

In [ ]:
SCRIPT=REPO/'cross_domain_reliability/stage11_major_revision/run_stage11d_best_checkpoint_sensitivity.py'
subprocess.run([sys.executable,str(SCRIPT),'--repo',str(REPO),'--stage5-root',str(RUN_ROOT),'--y-data',str(Y_DATA),'--c-data',str(C_DATA),'--out',str(OUT),'--original-group-summary',str(STAGE6/'stage6_group_summary.csv'),'--device','0','--batch','8','--imgsz','640','--ap-conf','0.001','--operating-conf','0.25','--nms-iou','0.70','--match-iou','0.50'],check=True)

In [ ]:
import pandas as pd
from IPython.display import display
display(pd.read_csv(OUT/'stage11d_best_vs_last_comparison.csv'))
print('STAGE11D_COMPLETE')